Python notebook for analysis of labeling results.

## Libraries and Data

In [1]:
import json
import pandas as pd
from tqdm import tqdm

from dataset_processing import CLIRENER_LABELS_V1, cwed4eta_process_json_file

import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime

import math
import numpy as np
from itertools import combinations

In [2]:
ls_file_path = "DATA/LABEL_STUDIO/project-30-at-2025-11-14-12-19-2a7464a5.json"

dataset = cwed4eta_process_json_file(ls_file_path)

### Helper Functions

In [4]:
def plot_tcel_anot(df: pd.DataFrame, output_dir: str = "PLOTS/ANNOTATION/") -> str:
    """
    Calculates and plots the Total Count of Entities per Label (TCEL_anot).

    The function generates a bar chart showing the total count for each entity
    label, sorted alphabetically. The plot is then saved to a specified
    directory with a formatted filename.

    Args:
        df (pd.DataFrame): A DataFrame with at least a 'label' column.
        output_dir (str): The directory where the plot will be saved.
                          Defaults to "PLOTS/ANNOTATION/".

    Returns:
        str: The full path to the saved plot image file.
    """
    # --- 1. Calculate Total Count of Entities per Label ---
    # Count occurrences and sort labels alphabetically for consistent order
    tcel_anot = df['label'].value_counts().sort_index()

    # --- 2. Plot the Histogram ---
    # Define consistent coloring based on sorted labels
    labels = tcel_anot.index
    colors = sns.color_palette('viridis', len(labels))
    color_map = dict(zip(labels, colors))

    plt.figure(figsize=(10, 6))
    bars = plt.bar(tcel_anot.index, tcel_anot.values, color=[color_map[label] for label in tcel_anot.index])

    plt.title('Total Count of Entities per Label (TCEL_anot)', fontsize=16)
    plt.xlabel('Entity Label', fontsize=12)
    plt.ylabel('Total Count', fontsize=12)
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    # Add counts on top of the bars for clarity
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval, int(yval), va='bottom', ha='center')

    plt.tight_layout()

    # --- 3. Save the Plot ---
    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Format the filename as per requirements: TCEL_{ddmmyy}_{number_of_labels}.png
    date_str = datetime.now().strftime("%d%m%y")
    num_labels = len(tcel_anot)
    filename = f"TCEL_{date_str}_{num_labels}.png"
    filepath = os.path.join(output_dir, filename)

    plt.savefig(filepath)
    plt.close() # Close the plot to free up memory

    print(f"Plot successfully saved to: {filepath}")

    return filepath


def plot_fdel_anot(df: pd.DataFrame, top_n: int = 10, output_dir: str = "PLOTS/ANNOTATION/") -> str:
    """
    Calculates and plots the Frequency Distribution of Entities per Label (FDEL_anot)
    using HORIZONTAL bar charts.

    For each label, this function finds the 'top_n' most frequent entity texts
    and plots them. The horizontal orientation is ideal for readability when
    entity names are long. All plots are arranged in a grid and saved to a
    single image file.

    Args:
        df (pd.DataFrame): A DataFrame with 'label' and 'entity_text' columns.
        top_n (int): The number of top entities to display for each label.
                     Defaults to 10.
        output_dir (str): The directory where the plot will be saved.
                          Defaults to "PLOTS/ANNOTATION/".

    Returns:
        str: The full path to the saved plot image file.
    """
    # --- 1. Data Preparation ---
    # Get a list of unique labels, sorted alphabetically for consistency
    labels = sorted(df['label'].unique())
    num_labels = len(labels)

    if num_labels == 0:
        print("DataFrame contains no labels to plot.")
        return ""

    # Define consistent coloring across all subplots
    colors = sns.color_palette('viridis', num_labels)
    color_map = dict(zip(labels, colors))

    # --- 2. Setup Subplots ---
    # Dynamically determine the grid size, aiming for 3 columns
    ncols = 3
    nrows = math.ceil(num_labels / ncols)

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(18, 5 * nrows))
    fig.suptitle(f'Top {top_n} Most Frequent Entities per Label (FDEL_anot)', fontsize=18, y=1.0)

    # Flatten axes array for easy iteration
    axes = axes.flatten() if num_labels > 1 else [axes]

    # --- 3. Generate a Plot for Each Label ---
    for idx, label in enumerate(labels):
        ax = axes[idx]
        
        # Filter for the current label and get the top N entity counts
        entity_counts = df[df['label'] == label]['entity_text'].value_counts().head(top_n)

        # Plot the data as a HORIZONTAL bar chart using kind='barh'
        # Sorting ascending ensures the most frequent entity is at the top
        entity_counts.sort_values(ascending=True).plot(
            kind='barh', # This is the key parameter for a horizontal plot
            ax=ax, 
            color=color_map[label]
        )

        ax.set_title(f"Label: '{label}'", fontsize=14)
        ax.set_xlabel("Frequency", fontsize=12) # X-axis is now Frequency
        ax.set_ylabel("Entity Text", fontsize=12) # Y-axis is now Entity Text
        ax.grid(axis='x', linestyle='--', alpha=0.7)

    # --- 4. Clean Up and Save ---
    # Hide any unused subplots in the grid
    for i in range(num_labels, len(axes)):
        axes[i].axis('off')

    plt.tight_layout(rect=[0, 0.03, 0.94, 0.97]) # Adjust layout

    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Format the filename: FDEL_{ddmmyy}_{number_of_labels}.png
    date_str = datetime.now().strftime("%d%m%y")
    filename = f"FDEL_{date_str}_{num_labels}.png"
    filepath = os.path.join(output_dir, filename)

    plt.savefig(filepath, dpi=300)
    plt.close() # Close the plot to free up memory

    print(f"Plot successfully saved to: {filepath}")

    return filepath


def plot_irbec_anot(df: pd.DataFrame, vmax: float = 0.2, output_dir: str = "PLOTS/ANNOTATION/") -> str:
    """
    Calculates and plots the Intersection Ratios Between Entity Classes (IRBEC_anot).

    This function computes a matrix of Jaccard similarity coefficients
    (intersection over union) for the sets of unique entity texts between
    every pair of labels. The resulting matrix is visualized as a heatmap.

    Args:
        df (pd.DataFrame): A DataFrame with 'label' and 'entity_text' columns.
        vmax (float): The maximum value for the heatmap's color scale, helping to
                      highlight small ratios. Defaults to 0.2.
        output_dir (str): The directory where the plot will be saved.
                          Defaults to "PLOTS/ANNOTATION/".

    Returns:
        str: The full path to the saved plot image file.
    """
    # --- 1. Prepare Data ---
    # Get unique labels sorted alphabetically for a consistent matrix order
    labels = sorted(df['label'].unique())
    num_labels = len(labels)

    if num_labels < 2:
        print("Cannot compute intersection ratios with fewer than two labels.")
        return ""

    # Create a dictionary mapping each label to the set of its unique entities
    # This is more efficient than re-calculating in the loop
    entity_sets = {
        label: set(df[df['label'] == label]['entity_text'].unique())
        for label in labels
    }

    # --- 2. Compute Pairwise Intersection Ratios ---
    # Initialize an empty DataFrame to store the ratios
    intersection_ratios = pd.DataFrame(index=labels, columns=labels, dtype=float)

    # Iterate over all unique pairs of labels
    for label1, label2 in combinations(labels, 2):
        entities1 = entity_sets[label1]
        entities2 = entity_sets[label2]
        
        # Calculate the length of the intersection and union
        intersection_len = len(entities1.intersection(entities2))
        union_len = len(entities1.union(entities2))
        
        # Compute Jaccard Index
        ratio = intersection_len / union_len if union_len > 0 else 0
        
        # The matrix is symmetric
        intersection_ratios.loc[label1, label2] = ratio
        intersection_ratios.loc[label2, label1] = ratio

    # Fill the diagonal with 1s (a label's intersection with itself is 100%)
    np.fill_diagonal(intersection_ratios.values, 1)

    # --- 3. Visualize the Heatmap ---
    plt.figure(figsize=(12, 10))
    sns.heatmap(
        intersection_ratios, 
        annot=True,          # Show the ratio values on the cells
        cmap="YlGnBu",       # A sequential colormap suitable for ratios
        fmt=".3f",           # Format numbers to 3 decimal places
        linewidths=0.5,
        vmin=0,              # Minimum value of the color bar is 0
        vmax=vmax,            # Set max value to focus on smaller overlaps
        annot_kws={'size': 4}  # <--- Change value fontsize
    )

    plt.title("Intersection Ratios Between Entity Classes (IRBEC_anot)", fontsize=16)
    plt.xlabel("Entity Label", fontsize=12)
    plt.ylabel("Entity Label", fontsize=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()

    # --- 4. Save the Plot ---
    os.makedirs(output_dir, exist_ok=True)
    
    # Format the filename: IRBEC_{ddmmyy}_{number_of_labels}.png
    date_str = datetime.now().strftime("%d%m%y")
    filename = f"IRBEC_{date_str}_{num_labels}.png"
    filepath = os.path.join(output_dir, filename)

    plt.savefig(filepath, dpi=300)
    plt.close()

    print(f"Plot successfully saved to: {filepath}")
    
    return filepath


def plot_lcoh_anot(df: pd.DataFrame, output_dir: str = "PLOTS/ANNOTATION/") -> str:
    """
    Calculates and plots the Label Co-occurrence Heatmap (LCOH).

    This function creates a matrix counting how many sentences contain pairs of
    entity labels. The diagonal represents the total number of sentences
    that contain a given label at least once. The off-diagonal cells represent
    the number of sentences where both the row and column labels appear together.

    Args:
        df (pd.DataFrame): A DataFrame with 'sentence_id' and 'label' columns.
        output_dir (str): The directory where the plot will be saved.
                          Defaults to "PLOTS/ANNOTATION/".

    Returns:
        str: The full path to the saved plot image file.
    """
    # --- 1. Data Preparation ---
    # Get unique labels sorted alphabetically for a consistent matrix order
    labels = sorted(df['label'].unique())
    num_labels = len(labels)

    if num_labels < 2:
        print("Cannot compute co-occurrence with fewer than two labels.")
        return ""

    # Initialize an empty DataFrame filled with zeros to store the counts
    co_occurrence_matrix = pd.DataFrame(0, index=labels, columns=labels, dtype=int)

    # --- 2. Compute Co-occurrence Counts ---
    # Group by sentence to analyze labels within each sentence
    for _, group in df.groupby('sentence_id'):
        # Get the set of unique labels present in the current sentence
        unique_labels_in_sentence = set(group['label'])

        # Increment the diagonal for each label present in the sentence
        for label in unique_labels_in_sentence:
            co_occurrence_matrix.loc[label, label] += 1
        
        # Increment the off-diagonal cells for each co-occurring pair
        # using combinations to get all unique pairs of labels
        for label1, label2 in combinations(unique_labels_in_sentence, 2):
            co_occurrence_matrix.loc[label1, label2] += 1
            co_occurrence_matrix.loc[label2, label1] += 1 # The matrix is symmetric

    # --- 3. Visualize the Heatmap ---
    plt.figure(figsize=(12, 10))
    sns.heatmap(
        co_occurrence_matrix,
        annot=True,               # Show the counts in the cells
        fmt='d',                  # Format as integer
        cmap='Blues',             # A good colormap for counts
        linewidths=0.5,
        annot_kws={'size': 10}    # Adjust font size for readability
    )

    plt.title("Label Co-occurrence Heatmap (LCOH)", fontsize=16)
    plt.xlabel("Entity Label", fontsize=12)
    plt.ylabel("Entity Label", fontsize=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()

    # --- 4. Save the Plot ---
    os.makedirs(output_dir, exist_ok=True)

    # Format the filename: LCOH_{ddmmyy}_{number_of_labels}.png
    date_str = datetime.now().strftime("%d%m%y")
    filename = f"LCOH_{date_str}_{num_labels}.png"
    filepath = os.path.join(output_dir, filename)

    plt.savefig(filepath, dpi=300)
    plt.close()

    print(f"Plot successfully saved to: {filepath}")

    return filepath


def summarize_irbec_ambiguities(df: pd.DataFrame, top_n: int = 4, output_dir: str = "PLOTS/ANNOTATION/") -> str:
    """
    Identifies top label ambiguities based on IRBEC analysis and generates a
    summarized CSV report.

    This function performs the following steps:
    1. Excludes the 'Other' label from the analysis.
    2. Calculates the Jaccard similarity (intersection over union) for the
       vocabulary of all remaining label pairs.
    3. For each label, it identifies the 'top_n' other labels with the highest
       (non-zero) similarity scores.
    4. Outputs the results to a CSV file with two columns: 'Labels' and
       'Ambiguities', where 'Ambiguities' is a comma-separated string of the
       most similar labels.

    Args:
        df (pd.DataFrame): The main DataFrame with 'label' and 'entity_text' columns.
        top_n (int): The number of top ambiguous labels to report for each
                     source label. Defaults to 4.
        output_dir (str): The directory where the CSV report will be saved.
                          Defaults to "PLOTS/ANNOTATION/".

    Returns:
        str: The full path to the saved CSV file.
    """
    # --- 1. Pre-process Data: Exclude the 'Other' label ---
    df_filtered = df[df['label'] != 'Other'].copy()
    
    # --- 2. Perform IRBEC Calculation on Filtered Data ---
    labels = sorted(df_filtered['label'].unique())
    num_labels = len(labels)

    if num_labels < 2:
        print("Not enough labels to compute ambiguities after filtering.")
        return ""

    entity_sets = {
        label: set(df_filtered[df_filtered['label'] == label]['entity_text'].unique())
        for label in labels
    }

    intersection_ratios = pd.DataFrame(index=labels, columns=labels, dtype=float)

    for label1, label2 in combinations(labels, 2):
        entities1 = entity_sets[label1]
        entities2 = entity_sets[label2]
        
        intersection_len = len(entities1.intersection(entities2))
        union_len = len(entities1.union(entities2))
        
        ratio = intersection_len / union_len if union_len > 0 else 0
        
        intersection_ratios.loc[label1, label2] = ratio
        intersection_ratios.loc[label2, label1] = ratio

    # --- 3. Process Ratios into the Summarized Format ---
    summary_data = []

    for source_label in intersection_ratios.index:
        # Get ratios, drop self-comparison, and filter out zero-value overlaps
        confusions = intersection_ratios.loc[source_label].drop(source_label)
        non_zero_confusions = confusions[confusions > 0]
        
        # Sort the remaining labels by their ratio and take the top N
        top_confusions = non_zero_confusions.sort_values(ascending=False).head(top_n)
        
        # Get the list of the ambiguous label names
        ambiguous_labels_list = top_confusions.index.tolist()
        
        # Format the list into the required comma-separated string
        ambiguities_str = ", ".join(ambiguous_labels_list)
        
        summary_data.append({
            'Labels': source_label,
            'Ambiguities': ambiguities_str
        })
            
    # Create the final report DataFrame
    report_df = pd.DataFrame(summary_data)

    # --- 4. Save the Report to CSV ---
    os.makedirs(output_dir, exist_ok=True)

    date_str = datetime.now().strftime("%d%m%y")
    filename = f"IRBEC_ambiguity_summary_{date_str}_{num_labels}.csv"
    filepath = os.path.join(output_dir, filename)

    report_df.to_csv(filepath, index=False)
    
    print(f"Ambiguity summary report successfully saved to: {filepath}")

    return filepath


## Loading

In [5]:
# This list will store our structured data
structured_data = []

# We iterate through each entry in your dataset
for i, entry in enumerate(dataset):
    sentence_id = f"sentence_{i+1}"  # Assign a unique ID to each sentence
    for entity in entry['entities']:
        structured_data.append({
            'sentence_id': sentence_id,
            'sentence_text': entry['text'],
            'entity_text': entity['text'],
            'label': entity['label'],
            'start_char': entity['start'],
            'end_char': entity['end']
        })

# We then create the DataFrame
df = pd.DataFrame(structured_data)

# Display the resulting DataFrame
print(df)

        sentence_id                                      sentence_text  \
0        sentence_1  This study aims to investigate the effect of N...   
1        sentence_1  This study aims to investigate the effect of N...   
2        sentence_1  This study aims to investigate the effect of N...   
3        sentence_1  This study aims to investigate the effect of N...   
4        sentence_1  This study aims to investigate the effect of N...   
...             ...                                                ...   
9523  sentence_1228  This means that the time required to acquire a...   
9524  sentence_1228  This means that the time required to acquire a...   
9525  sentence_1228  This means that the time required to acquire a...   
9526  sentence_1228  This means that the time required to acquire a...   
9527  sentence_1228  This means that the time required to acquire a...   

                        entity_text                  label  start_char  \
0                             study  

## Plotting

In [6]:
# TCEL
plot_tcel_anot(df)

# FDEL
plot_fdel_anot(df)

# IRBEC
plot_irbec_anot(df)
summarize_irbec_ambiguities(df)

# LCOH
plot_lcoh_anot(df)

Plot successfully saved to: PLOTS/ANNOTATION/TCEL_141125_28.png
Plot successfully saved to: PLOTS/ANNOTATION/FDEL_141125_28.png
Plot successfully saved to: PLOTS/ANNOTATION/IRBEC_141125_28.png
Ambiguity summary report successfully saved to: PLOTS/ANNOTATION/IRBEC_ambiguity_summary_141125_27.csv
Plot successfully saved to: PLOTS/ANNOTATION/LCOH_141125_28.png


'PLOTS/ANNOTATION/LCOH_141125_28.png'